In [1]:
import sys
from typing import List
import fitz  # PyMuPDF

from src.Utils.logger_setup import get_log, track_performance
from src.Utils.exception_handler import CustomException

# Initialize logger
logger = get_log(__name__)


@track_performance
def ingest_pdf(file_path: str) -> List[str]:
    """
    Extract raw text from a PDF page by page using PyMuPDF with logging and error handling.
    """
    logger.info(f"Starting PDF ingestion for file: {file_path}")

    try:
        doc = fitz.open(file_path)
        pages: List[str] = []

        for i, page in enumerate(doc):
            text = page.get_text("text") or ""

            if text.strip():
                pages.append(text)
            else:
                logger.warning(f"No text found on page {i + 1} of {file_path}")

        doc.close()

        logger.info(f"Successfully extracted {len(pages)} pages from {file_path}")
        return pages

    except Exception as e:
        logger.error(f"Error occurred while ingesting PDF: {str(e)}")
        raise CustomException(e, sys)

In [2]:
import sys
import re
import binascii
import unicodedata
from hashlib import blake2b
from typing import List, Optional, Set

import fitz  # PyMuPDF

from src.Utils.logger_setup import get_log, track_performance
from src.Utils.exception_handler import CustomException

logger = get_log(__name__)


# =========================
# PDF INGESTION (PyMuPDF)
# =========================
@track_performance
def ingest_pdf(file_path: str) -> List[str]:
    """
    Extract raw text from PDF page by page using PyMuPDF.
    Output is a list of page texts.
    """
    logger.info(f"Starting PDF ingestion for file: {file_path}")

    try:
        doc = fitz.open(file_path)
        pages: List[str] = []

        for i, page in enumerate(doc):
            text = page.get_text("text") or ""

            if text.strip():
                pages.append(text)
            else:
                logger.warning(f"No text found on page {i + 1} of {file_path}")

        doc.close()

        logger.info(f"Successfully extracted {len(pages)} pages from {file_path}")
        return pages

    except Exception as e:
        logger.error(f"Error occurred while ingesting PDF: {str(e)}")
        raise CustomException(e, sys)


# =========================
# TEXT GUARDRAILS
# =========================
class TextGuardrails:
    def __init__(self, semantic_threshold: float = 0.85, num_perm: int = 128):
        self.threshold = semantic_threshold
        self.num_perm = num_perm
        self.exact_hashes: Set[str] = set()
        self.minhash_registry: List[List[int]] = []

    # ---------- Normalization ----------
    @staticmethod
    def step_normalize_unicode(text: str) -> str:
        return unicodedata.normalize("NFKC", text)

    # ---------- PII Masking ----------
    @staticmethod
    def step_mask_pii(text: str) -> str:
        patterns = {
            r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+": "[EMAIL_MASKED]",
            r"(\+91[\-\s]?)?[6-9]\d{9}": "[PHONE_MASKED]",
            r"\b\d{4}\s?\d{4}\s?\d{4}\b": "[AADHAAR_MASKED]",
        }

        for regex, mask in patterns.items():
            text = re.sub(regex, mask, text)

        return text

    # ---------- Whitespace cleanup ----------
    @staticmethod
    def step_clean_whitespace(text: str) -> str:
        return " ".join(text.split())

    # ---------- Duplicate detection ----------
    def is_duplicate(self, text: str) -> bool:
        """Checks if a sentence is an exact or semantic duplicate."""

        h = blake2b(text.encode(), digest_size=16).hexdigest()
        if h in self.exact_hashes:
            return True

        clean = "".join(text.lower().split())
        if len(clean) < 10:
            return False

        shingles = {clean[i:i + 5] for i in range(len(clean) - 4)}
        if not shingles:
            return False

        current_sig = [
            min(binascii.crc32(f"{s}{j}".encode()) & 0xffffffff for s in shingles)
            for j in range(self.num_perm)
        ]

        for existing_sig in self.minhash_registry:
            match_count = sum(a == b for a, b in zip(current_sig, existing_sig))
            if (match_count / self.num_perm) >= self.threshold:
                return True

        self.exact_hashes.add(h)
        self.minhash_registry.append(current_sig)
        return False

    # ---------- Main pipeline ----------
    def apply(self, text: str) -> Optional[str]:
        """
        Processes text:
        - normalize
        - mask PII
        - clean whitespace
        - remove duplicate sentences
        """

        try:
            text = self.step_normalize_unicode(text)
            text = self.step_mask_pii(text)
            text = self.step_clean_whitespace(text)

            sentences = re.split(r'(?<=[.!?]) +', text)
            unique_sentences = []

            for sent in sentences:
                sent_clean = sent.strip()

                if sent_clean and not self.is_duplicate(sent_clean):
                    unique_sentences.append(sent_clean)
                else:
                    logger.debug(f"Filtered duplicate sentence: {sent_clean[:40]}...")

            return " ".join(unique_sentences) if unique_sentences else None

        except Exception as e:
            raise CustomException(e, sys)


# =========================
# PIPELINE: PDF → CLEAN TEXT
# =========================
def process_pdf(file_path: str, guardrails: TextGuardrails) -> List[str]:
    """
    Full pipeline:
    1. Extract PDF text (PyMuPDF)
    2. Apply TextGuardrails per page
    """

    pages = ingest_pdf(file_path)
    processed_pages: List[str] = []

    for i, page_text in enumerate(pages):
        cleaned = guardrails.apply(page_text)

        if cleaned:
            processed_pages.append(cleaned)
        else:
            logger.warning(f"Page {i + 1} removed after guardrails filtering")

    return processed_pages

In [11]:
data = process_pdf(r"F:\Loan_Details_RAG_System1\Data\loan_products_final.pdf", TextGuardrails())

No text found on page 35 of F:\Loan_Details_RAG_System1\Data\loan_products_final.pdf


In [12]:
data

['SecureSave Loan Against Approved Securities (SLAAS) Product Overview SecureSave Loan Against Approved Securities (SLAAS) is a secured credit facility offered to resident individuals against the pledge or assignment of approved financial securities such as National Savings Certificates (NSC), Kisan Vikas Patra (KVP), Life Insurance Policies, and Government/Relief Bonds. The product is designed to meet both productive financial needs and personal contingency requirements, while ensuring safety of funds through high-quality collateral. Nature of Facility The facility is available in the form of Demand Loan, Term Loan, or Overdraft. The loan is generally sanctioned for a period of 12 months, subject to annual review, and may continue up to the maturity of the underlying security in case of overdraft facilities. Eligibility The facility is available to resident individuals aged 21 years and above. Advances to third parties are strictly not permitted. Eligible Securities Loans can be avail

In [3]:
import json
import asyncio
import time
from collections import Counter
from typing import List, Dict

import ollama
import nest_asyncio

nest_asyncio.apply()


class BatchPDFKeywordPipeline:
    def __init__(self, model_name="qwen2.5:7b", batch_size=3, max_concurrency=2, rate_limit=2):
        self.model_name = model_name
        self.batch_size = batch_size
        self.steps = []
        self.semaphore = asyncio.Semaphore(max_concurrency)
        self.rate_limit = rate_limit
        self.last_call_time = 0

    # -----------------------------
    # PIPELINE CONTROL
    # -----------------------------
    def add_step(self, func):
        self.steps.append(func)

    async def run(self, input_data):
        data = input_data
        for step in self.steps:
            print(f"Running: {step.__name__}")

            if asyncio.iscoroutinefunction(step):
                data = await step(self, data)
            else:
                data = step(self, data)

        return data

    async def rate_limiter(self):
        min_interval = 1.0 / self.rate_limit
        elapsed = time.time() - self.last_call_time

        if elapsed < min_interval:
            await asyncio.sleep(min_interval - elapsed)

        self.last_call_time = time.time()

    # -----------------------------
    # PROMPT
    # -----------------------------
    def build_batch_prompt(self, batch):
        return f"""
You are a Retail Banking Domain Expert.

TASK:
Assign EXACTLY 5 banking keywords per ID.

RULES:
- Use text primarily
- Use topic + section context if needed
- Always return 5 keywords
- No empty values

OUTPUT:
{{
  "ID": ["k1","k2","k3","k4","k5"]
}}

INPUT:
{json.dumps(batch, indent=2)}
"""

    # -----------------------------
    # LLM CALL
    # -----------------------------
    async def process_batch(self, batch):
        async with self.semaphore:
            await self.rate_limiter()

            try:
                response = await asyncio.to_thread(
                    ollama.chat,
                    model=self.model_name,
                    format="json",
                    messages=[
                        {
                            "role": "system",
                            "content": "Return ONLY JSON. Each key must map to exactly 5 strings."
                        },
                        {
                            "role": "user",
                            "content": self.build_batch_prompt(batch)
                        }
                    ],
                    options={"temperature": 0.2}
                )

                raw = response["message"]["content"].strip()
                return json.loads(raw)

            except Exception as e:
                print(f"Batch error: {e}")
                return {}

    # =========================================================
    # STEP 1 (UPDATED): INPUT = CLEANED PAGES (NOT FITZ SPANS)
    # =========================================================
    def step_prepare_spans_from_pages(self, pages: List[str]):
        """
        Converts cleaned pages into pseudo-spans so existing logic works.
        Each line becomes a 'span' with fake size heuristics disabled.
        """

        spans = []
        page_id = 0

        for page_text in pages:
            page_id += 1

            lines = [l.strip() for l in page_text.split("\n") if l.strip()]

            for line in lines:
                spans.append({
                    "page_id": page_id,
                    "text": line,
                    "size": 12.0  # neutral size since formatting is lost after cleaning
                })

        return spans

    # -----------------------------
    # STEP 2: PARSE HIERARCHY
    # -----------------------------
    def step_parse_hierarchy(self, spans):
        sizes = [s["size"] for s in spans]
        freq = Counter(sizes)
        sorted_sizes = sorted(freq.keys(), reverse=True)

        section_font = sorted_sizes[0] if sorted_sizes else 12.0
        topic_font = sorted_sizes[0] if len(sorted_sizes) < 2 else sorted_sizes[1]

        results = []
        current_section, current_topic = None, None
        sc, tc = 0, 0

        for span in spans:
            text, size = span["text"], span["size"]

            if size == section_font:
                sc += 1
                tc = 0

                current_section = {
                    "section_id": f"S{sc:03d}",
                    "section_name": text,
                    "topics": []
                }
                results.append(current_section)
                current_topic = None

            elif size == topic_font and current_section:
                tc += 1

                current_topic = {
                    "topic_id": f"{current_section['section_id']}_T{tc:03d}",
                    "topic_name": text,
                    "lines": []
                }
                current_section["topics"].append(current_topic)

            elif current_topic:
                current_topic["lines"].append(text)

        return results

    # -----------------------------
    # STEP 3: FLATTEN
    # -----------------------------
    def step_merge_topics_sequentially(self, data):
        flattened_topics = []
        paragraph_counter = 0

        for section in data:
            for topic in section["topics"]:
                paragraph_counter += 1

                raw_text = " ".join(topic["lines"]).strip()
                clean_text = " ".join(raw_text.split())

                flattened_topics.append({
                    "page_id": paragraph_counter,
                    "paragraph_id": f"P_{paragraph_counter:03d}",
                    "topic_id": topic["topic_id"],
                    "topic_name": topic["topic_name"],
                    "section_id": section["section_id"],
                    "section_name": section["section_name"],
                    "text": clean_text
                })

        return flattened_topics

    # -----------------------------
    # STEP 4: KEYWORD ENRICHMENT
    # -----------------------------
    async def step_enrich_topic_keywords(self, flattened_topics):
        batches = [
            flattened_topics[i:i + self.batch_size]
            for i in range(0, len(flattened_topics), self.batch_size)
        ]

        tasks = []

        for batch in batches:
            payload = [
                {
                    "id": t["topic_id"],
                    "topic_name": t["topic_name"],
                    "section_name": t["section_name"],
                    "text": t["text"][:2000]
                }
                for t in batch
            ]

            tasks.append(asyncio.create_task(self.process_batch(payload)))

        all_keywords = {}
        batch_results = await asyncio.gather(*tasks)

        for res in batch_results:
            all_keywords.update(res)

        for topic in flattened_topics:
            tid = topic["topic_id"]
            topic["keywords"] = all_keywords.get(tid, [])

            while len(topic["keywords"]) < 5:
                topic["keywords"].append("N/A")

        return flattened_topics

In [7]:
import asyncio
import json
from typing import Dict, Any


async def run_pdf_keyword_pipeline(
    file_path: str,
    output_path: str = "final_output.json"
) -> Dict[str, Any]:
    """
    Full pipeline:
    PDF → Clean Pages → Structure → Keywords → JSON output + file save
    """

    try:
        # -----------------------------
        # STEP 1: PDF → CLEAN PAGES
        # -----------------------------
        guardrails = TextGuardrails()
        pages = process_pdf(file_path, guardrails)

        if not pages:
            result = {
                "status": "failed",
                "reason": "No extractable text found in PDF",
                "data": []
            }

            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(result, f, indent=4, ensure_ascii=False)

            return result

        # -----------------------------
        # STEP 2: INIT PIPELINE
        # -----------------------------
        pipeline = BatchPDFKeywordPipeline(
            batch_size=3,
            max_concurrency=2
        )

        pipeline.add_step(BatchPDFKeywordPipeline.step_prepare_spans_from_pages)
        pipeline.add_step(BatchPDFKeywordPipeline.step_parse_hierarchy)
        pipeline.add_step(BatchPDFKeywordPipeline.step_merge_topics_sequentially)
        pipeline.add_step(BatchPDFKeywordPipeline.step_enrich_topic_keywords)

        # -----------------------------
        # STEP 3: RUN PIPELINE
        # -----------------------------
        result_data = await pipeline.run(pages)

        result = {
            "status": "success",
            "file_path": file_path,
            "total_pages": len(pages),
            "total_topics": len(result_data),
            "data": result_data
        }

        # -----------------------------
        # STEP 4: SAVE JSON
        # -----------------------------
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(result_data, f, indent=4, ensure_ascii=False)

        return result

    except Exception as e:
        error_result = {
            "status": "error",
            "error": str(e),
            "data": []
        }

        # save error too
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(error_result, f, indent=4, ensure_ascii=False)

        return error_result

In [8]:
import asyncio

file_path = r"C:\Users\prasa\Downloads\loan_products_final.pdf"

result = asyncio.run(
    run_pdf_keyword_pipeline(
        file_path,
        output_path="final_merged_keywords.json"
    )
)

print(result["status"])

No text found on page 35 of C:\Users\prasa\Downloads\loan_products_final.pdf


Running: step_prepare_spans_from_pages
Running: step_parse_hierarchy
Running: step_merge_topics_sequentially
Running: step_enrich_topic_keywords
success


In [1]:
import fitz
import json
import asyncio
import time
import sys
import re
import binascii
import unicodedata
import ollama
import nest_asyncio
from typing import List, Set, Optional, Dict
from hashlib import blake2b
from collections import Counter

# Project Utility Imports
try:
    from src.Utils.logger_setup import get_log, track_performance
    from src.Utils.exception_handler import CustomException
    logger = get_log(__name__)
except ImportError:
    import logging
    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger(__name__)
    def track_performance(func): return func
    class CustomException(Exception):
        def __init__(self, e, sys_ref): super().__init__(e)

nest_asyncio.apply()

# ---------------------------------------------------------
# TEXT GUARDRAILS (Sentence-Level Deduplication)
# ---------------------------------------------------------
class TextGuardrails:
    def __init__(self, semantic_threshold: float = 0.85, num_perm: int = 64):
        self.threshold = semantic_threshold
        self.num_perm = num_perm
        self.exact_hashes: Set[str] = set()
        self.minhash_registry: List[List[int]] = []

    def _get_minhash_signature(self, text: str) -> List[int]:
        """Generates a MinHash signature for fuzzy deduplication."""
        clean = "".join(text.lower().split())
        shingles = {clean[i:i + 5] for i in range(len(clean) - 4)}
        if not shingles:
            return []
        
        return [
            min(binascii.crc32(f"{s}{j}".encode()) & 0xffffffff for s in shingles)
            for j in range(self.num_perm)
        ]

    def is_duplicate(self, text: str) -> bool:
        """Checks for exact and semantic (fuzzy) duplicates."""
        # 1. Exact Match
        h = blake2b(text.encode(), digest_size=16).hexdigest()
        if h in self.exact_hashes:
            return True
        
        # 2. Semantic Match
        if len(text) < 15: return False # Ignore very short phrases
        
        current_sig = self._get_minhash_signature(text)
        if not current_sig: return False

        for existing_sig in self.minhash_registry:
            match_count = sum(a == b for a, b in zip(current_sig, existing_sig))
            if (match_count / self.num_perm) >= self.threshold:
                return True

        # Save unique data
        self.exact_hashes.add(h)
        self.minhash_registry.append(current_sig)
        return False

    def apply(self, text: str) -> Optional[str]:
        """PII Masking, Normalization, and Deduplication."""
        # Normalization
        text = unicodedata.normalize("NFKC", text)
        
        # Mask PII
        text = re.sub(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+", "[EMAIL_MASKED]", text)
        text = re.sub(r"(\+91[\-\s]?)?[6-9]\d{9}", "[PHONE_MASKED]", text)
        text = re.sub(r"\b\d{4}\s?\d{4}\s?\d{4}\b", "[AADHAAR_MASKED]", text)
        
        # Whitespace
        text = " ".join(text.split())

        # Sentence deduplication
        sentences = re.split(r'(?<=[.!?]) +', text)
        unique_sentences = [s.strip() for s in sentences if s.strip() and not self.is_duplicate(s.strip())]
        
        return " ".join(unique_sentences) if unique_sentences else None

# ---------------------------------------------------------
# BATCH PDF PIPELINE
# ---------------------------------------------------------
class BatchPDFKeywordPipeline:
    def __init__(self, model_name="qwen2.5:7b", batch_size=6):
        self.model_name = model_name
        self.batch_size = batch_size
        self.semaphore = asyncio.Semaphore(2)

    async def process_batch(self, batch: List[Dict]) -> Dict[str, List[str]]:
        async with self.semaphore:
            try:
                # Optimized prompt for Retail Banking Domain Expert
                prompt = (
                    "You are a Retail Banking Domain Expert.\n"
                    "TASK: Assign EXACTLY 5 banking keywords per topic_id.\n"
                    "RULES:\n"
                    "- Use provided text primarily.\n"
                    "- Use topic + section context if needed.\n"
                    "- Always return 5 keywords. No empty values.\n"
                    f"INPUT DATA: {json.dumps(batch)}"
                )
                
                response = await asyncio.to_thread(
                    ollama.chat,
                    model=self.model_name,
                    format="json",
                    messages=[{"role": "user", "content": prompt}],
                    options={"temperature": 0.1}
                )
                
                content = response["message"]["content"].strip()
                if "```json" in content:
                    content = content.split("```json")[1].split("```")[0].strip()
                
                return json.loads(content)
            except Exception as e:
                logger.error(f"LLM Error: {e}")
                return {}

    def extract_hierarchy(self, pdf_path: str) -> List[Dict]:
        """Extracts Section and Topic hierarchy based on font sizes."""
        doc = fitz.open(pdf_path)
        spans = []
        for page in doc:
            for block in page.get_text("dict")["blocks"]:
                if "lines" not in block: continue
                for line in block["lines"]:
                    for span in line["spans"]:
                        if span["text"].strip():
                            spans.append({"text": span["text"].strip(), "size": round(span["size"], 1)})
        
        sizes = sorted(Counter([s["size"] for s in spans]).keys(), reverse=True)
        if len(sizes) < 2: return []
        
        section_f, topic_f = sizes[0], sizes[1]
        results, cur_sec_name, cur_sec_id = [], "", ""
        s_idx, t_idx = 0, 0

        for s in spans:
            if s["size"] == section_f:
                s_idx += 1
                cur_sec_name = s["text"]
                cur_sec_id = f"S{s_idx:03}"
                t_idx = 0 
            elif s["size"] == topic_f and cur_sec_id:
                t_idx += 1
                results.append({
                    "section_id": cur_sec_id,
                    "section_name": cur_sec_name,
                    "topic_id": f"{cur_sec_id}_T{t_idx:03}",
                    "topic_name": s["text"],
                    "raw_lines": []
                })
            elif results:
                results[-1]["raw_lines"].append(s["text"])
        
        doc.close()
        return results

# ---------------------------------------------------------
# MASTER ORCHESTRATOR
# ---------------------------------------------------------
@track_performance
async def run_master_pipeline(pdf_path: str):
    start_time = time.time()
    logger.info(f" PIPELINE STARTED | File: {pdf_path}")

    pipeline = BatchPDFKeywordPipeline()
    guardrails = TextGuardrails()

    # -----------------------------------------------------
    # 1. EXTRACTION
    # -----------------------------------------------------
    logger.info(" Step 1: Extracting PDF hierarchy...")
    extraction_start = time.time()

    extracted_data = pipeline.extract_hierarchy(pdf_path)

    logger.info(
        f" Extraction complete | Sections+Topics: {len(extracted_data)} | "
        f"Time: {time.time() - extraction_start:.2f}s"
    )

    if not extracted_data:
        logger.warning(" No structured data extracted. Exiting early.")
        return []

    # -----------------------------------------------------
    # 2. CLEANING + PARAGRAPH BUILDING
    # -----------------------------------------------------
    logger.info(" Step 2: Cleaning & structuring text...")
    clean_start = time.time()

    processed_items = []
    p_idx = 1
    skipped = 0

    for idx, item in enumerate(extracted_data, 1):
        raw_text = " ".join(item.pop("raw_lines"))

        logger.debug(f"[{idx}] Raw length: {len(raw_text)}")

        clean_content = guardrails.apply(raw_text)

        if clean_content:
            item["para_no"] = f"P_{p_idx:03}"
            item["text"] = clean_content
            processed_items.append(item)

            logger.debug(
                f"[{idx}]  Cleaned | Para: {item['para_no']} | "
                f"Length: {len(clean_content)}"
            )
            p_idx += 1
        else:
            skipped += 1
            logger.debug(f"[{idx}] ❌ Skipped (empty/duplicate)")

    logger.info(
        f" Cleaning complete | Valid: {len(processed_items)} | "
        f"Skipped: {skipped} | Time: {time.time() - clean_start:.2f}s"
    )

    # -----------------------------------------------------
    # 3. BATCH KEYWORD GENERATION
    # -----------------------------------------------------
    logger.info(" Step 3: Generating keywords via LLM...")

    final_output = []
    total_batches = (len(processed_items) + pipeline.batch_size - 1) // pipeline.batch_size

    for batch_idx, i in enumerate(range(0, len(processed_items), pipeline.batch_size), 1):
        batch = processed_items[i : i + pipeline.batch_size]

        logger.info(
            f" Batch {batch_idx}/{total_batches} | "
            f"Items: {len(batch)}"
        )

        batch_start = time.time()

        llm_batch = [
            {"topic_id": x["topic_id"], "text": x["text"][:600]}
            for x in batch
        ]

        keyword_results = await pipeline.process_batch(llm_batch)

        logger.debug(f"Batch {batch_idx} LLM response keys: {list(keyword_results.keys())}")

        for item in batch:
            topic_id = item["topic_id"]

            if topic_id in keyword_results:
                item["keywords"] = keyword_results[topic_id]
                logger.debug(f" Keywords assigned | {topic_id}")
            else:
                item["keywords"] = ["Banking", "Compliance", "Retail", "Loan", "Service"]
                logger.warning(f"⚠ Default keywords used | {topic_id}")

            final_output.append(item)

        logger.info(
            f" Batch {batch_idx} complete | "
            f"Time: {time.time() - batch_start:.2f}s"
        )

    # -----------------------------------------------------
    # 4. SAVE OUTPUT
    # -----------------------------------------------------
    logger.info(" Step 4: Saving output to JSON...")
    save_start = time.time()

    output_file = "retail_banking_structured.json"
    with open(output_file, "w") as f:
        json.dump(final_output, f, indent=4)

    logger.info(
        f" Saved: {output_file} | Records: {len(final_output)} | "
        f"Time: {time.time() - save_start:.2f}s"
    )

    # -----------------------------------------------------
    # FINAL SUMMARY
    # -----------------------------------------------------
    total_time = time.time() - start_time

    logger.info(" PIPELINE SUMMARY")
    logger.info(f"   Total Topics Extracted : {len(extracted_data)}")
    logger.info(f"   Final Paragraphs      : {len(final_output)}")
    logger.info(f"   Total Time            : {total_time:.2f}s")

    logger.info(" PIPELINE COMPLETED SUCCESSFULLY")

    return final_output

if __name__ == "__main__":
    asyncio.run(run_master_pipeline("F:\\Loan_Details_RAG_System1\\Data\\loan_products_final.pdf"))

KeyboardInterrupt: 